# Gemma 4 L4x4 paired agent experiment

This private notebook compares the scored V3 candidate with a focused V4 prompt/tool variant on the same eight public development issues using the official competition Gemma 4 31B W4A16 model on Kaggle's four-L4 runtime. V3 has a public score of 0.08. The local development panel is a small tuning signal; it does not guarantee a higher Kaggle score.

The model receives only the issue and repository through the official evaluator. Reference patches are not passed to the agent. No adapter is used because a current competition report found that patched vLLM can load Gemma 4 LoRA weights while applying no logit change. Internet is disabled.


In [ ]:
from pathlib import Path
import glob, importlib, json, os, shutil, subprocess, sys

os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

WORK = Path('/kaggle/working')
DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
MODEL_PATH = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2')
assert (DATA_DIR / 'tasks.jsonl').is_file(), DATA_DIR
assert MODEL_PATH.exists(), MODEL_PATH
assert WHEELHOUSE_DIR.is_dir(), WHEELHOUSE_DIR

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                     capture_output=True, text=True, check=True).stdout.splitlines()
print('Visible GPUs:', gpu)
assert len(gpu) == 4 and all('L4' in name for name in gpu), gpu

# Install the competition's pinned offline wheelhouse, following the official starter.
for pattern in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth',
                '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for pth in glob.glob(pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass
tmp_whl = Path('/tmp/gemma4-wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for wheel in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in wheel.name.lower():
        continue
    renamed = wheel.name.replace('cu128', '+cu128') if ('cu128' in wheel.name and '+' not in wheel.name) else wheel.name
    link = tmp_whl / renamed
    if not link.exists():
        link.symlink_to(wheel)
wheels = sorted(str(p) for p in tmp_whl.glob('*.whl'))
assert wheels, f'No official wheels found in {WHEELHOUSE_DIR}'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index',
                '--no-deps', '--force-reinstall', *wheels], check=True)
importlib.invalidate_caches()
print('Installed official wheelhouse:', len(wheels), 'wheels')


## Agent arms

The first pair isolates the per-task time cap. The lean candidate also removes graph and skill calls, since public inspection reports missing graph/embedding files for the 129 public tasks, and it directs Gemma to attempt an evidence-based source edit earlier. The public scored bundle is included as a reference arm.


In [ ]:
VARIANT_FILES = {'lean_v3_8m': {'agent.yaml': 'name: gemma4_swe_agent_v2\n'
                              'model: gemma-4-31b-it-qat-w4a16-ct\n'
                              'description: Autonomous software engineer that localizes and '
                              'repairs repository issues with concise evidence checks.\n'
                              'instruction: !include prompts/system.md\n'
                              'tools:\n'
                              '  - run_command\n'
                              '  - get_status\n'
                              '  - read_file\n'
                              '  - edit_file\n'
                              '  - write_file\n'
                              '  - submit_patch\n'
                              '  - agent_tool:\n'
                              '      config_path: sub_agents/code_analyzer.yaml\n'
                              '      skip_summarization: true\n'
                              'generate_content_config: !include configs/sampling.yaml\n',
                'configs/sampling.yaml': 'temperature: 0.2\n'
                                         'top_p: 0.95\n'
                                         'top_k: 40\n'
                                         'max_output_tokens: 4096\n'
                                         'thinking_config:\n'
                                         '  thinking_budget: 2048\n'
                                         '  include_thoughts: false\n',
                'eval_config.yaml': 'evaluation:\n'
                                    '  timeout_seconds: 120\n'
                                    '  max_tool_calls: 28\n'
                                    '  max_time_minutes: 8\n'
                                    '  max_turns: 24\n',
                'prompts/system.md': 'You are an autonomous senior software engineer fixing one '
                                     'issue in the repository mounted at `/workspace`.\n'
                                     '\n'
                                     '## Required finish\n'
                                     'The task is not complete until you invoke `submit_patch()` '
                                     'as your final tool call. A written explanation does not '
                                     'submit a patch. Make the strongest evidence-based '
                                     'production-code change you can and submit it even when a '
                                     'test cannot run or still fails. Do not finish with an empty '
                                     'patch merely because verification is incomplete.\n'
                                     '\n'
                                     '## Tests are strictly read-only\n'
                                     'Never call `edit_file` or `write_file` on a test path. This '
                                     'includes any path under `tests/`, any `test_*.py` or '
                                     '`*_test.py` file, `conftest.py`, and `pytest.ini`. Do not '
                                     'create, delete, rename, or change tests or fixtures, even to '
                                     'reproduce behavior. Before submission, inspect the '
                                     'changed-path list. If a test path was changed, run `git '
                                     'restore -- <that exact test path>` and recheck the diff; '
                                     'submit production code only. The benchmark applies its own '
                                     'tests after you finish.\n'
                                     '\n'
                                     '## Work quickly from evidence\n'
                                     '1. Use the supplied workspace layout and issue. If the issue '
                                     'names a file, symbol, or error string, inspect that location '
                                     'first; otherwise run one bounded `rg --files` or literal `rg '
                                     '-n -F` search.\n'
                                     '2. Read the smallest useful implementation range and form '
                                     'one concrete failure hypothesis. Do not search git history. '
                                     'Do not run tests before the first edit unless a tiny '
                                     'reproducer is already available.\n'
                                     '3. If two quick source checks do not identify a plausible '
                                     'location, call `code_analyzer` once with the issue and '
                                     'evidence. It is read-only; confirm its suggestion in '
                                     'source.\n'
                                     '4. Make one minimal production-code edit by about the eighth '
                                     'model/tool step. Use `edit_file(filepath, old_string, '
                                     'new_string)` with text copied exactly from source. Retry '
                                     'once after rereading a smaller range if it fails. Never '
                                     'modify tests, fixtures, `conftest.py`, `pytest.ini`, '
                                     'packaging, or runner configuration.\n'
                                     '5. Inspect the diff and run at most one short, relevant '
                                     'check. If that check finds a repairable error, make one '
                                     'focused correction. Never run a full suite, repeat an '
                                     'unchanged check, install packages, or use the network.\n'
                                     '6. Inspect the changed paths and restore any test-path '
                                     'changes using `git restore -- <exact path>`. Confirm only '
                                     'intended production files remain.\n'
                                     '7. Call `submit_patch()` no later than model/tool step 18 or '
                                     'when 60 seconds remain, whichever comes first. Do not make '
                                     'another tool call after it. If unsure, submit the best '
                                     'coherent source patch rather than spending the remaining '
                                     'budget on more searches.\n'
                                     '\n'
                                     '## Repository navigation\n'
                                     '- Prefer `rg --files` and literal `rg -n -F` searches; keep '
                                     'output bounded.\n'
                                     '- Use graph tools only if they are available and a '
                                     'source-level symbol has already been identified. Pass exact '
                                     'symbol names, not prose. If graph data is missing, continue '
                                     'with source search and do not repeat the same graph query.\n'
                                     '- Avoid broad directory dumps, repeated reads, and large '
                                     'file rewrites. Keep tool payloads concise. Check '
                                     '`get_status` only when the remaining budget could change the '
                                     'next action.\n'
                                     '\n'
                                     '## Repair quality\n'
                                     '- Preserve public interfaces and nearby conventions; handle '
                                     "the issue's stated edge cases.\n"
                                     '- Never invent verification results. Check the final diff '
                                     'before submitting.\n'
                                     '- A focused check failure is evidence for at most one '
                                     'correction; it is not a reason to delay `submit_patch()`.\n',
                'sub_agents/analyzer.md': 'You are a read-only code localizer for one repository '
                                          'issue. Use the task statement and current source only. '
                                          'Return at most 180 words with: likely implementation '
                                          'files and exact symbols; the source evidence supporting '
                                          'the location; one useful existing test or narrow check; '
                                          'and remaining uncertainty. Start with bounded literal '
                                          '`rg` searches and short file reads. Do not edit files, '
                                          'run broad tests, inspect git history, or claim a repair '
                                          'is verified. If graph data is unavailable, continue '
                                          'with ordinary source inspection.\n',
                'sub_agents/code_analyzer.yaml': 'name: code_analyzer\n'
                                                 'model: gemma-4-31b-it-qat-w4a16-ct\n'
                                                 'description: Read-only repository localizer that '
                                                 'reports likely implementation symbols and a '
                                                 'narrow verification target.\n'
                                                 'instruction: !include analyzer.md\n'
                                                 'tools:\n'
                                                 '  - run_command\n'
                                                 '  - read_file\n'
                                                 'generate_content_config:\n'
                                                 '  temperature: 0.1\n'
                                                 '  top_p: 0.95\n'
                                                 '  max_output_tokens: 1536\n'
                                                 '  thinking_config:\n'
                                                 '    include_thoughts: false\n'},
 'lean_v4_6m': {'agent.yaml': 'name: gemma4_swe_agent_v4\n'
                              'model: gemma-4-31b-it-qat-w4a16-ct\n'
                              'description: Focused Gemma SWE agent that makes an evidence-based '
                              'production fix and submits promptly.\n'
                              'instruction: !include prompts/system.md\n'
                              'tools:\n'
                              '  - run_command\n'
                              '  - read_file\n'
                              '  - edit_file\n'
                              '  - submit_patch\n'
                              'generate_content_config: !include configs/sampling.yaml\n',
                'configs/sampling.yaml': 'temperature: 0.2\n'
                                         'top_p: 0.95\n'
                                         'top_k: 40\n'
                                         'max_output_tokens: 4096\n'
                                         'thinking_config:\n'
                                         '  thinking_budget: 2048\n'
                                         '  include_thoughts: false\n',
                'eval_config.yaml': 'evaluation:\n'
                                    '  timeout_seconds: 120\n'
                                    '  max_tool_calls: 16\n'
                                    '  max_time_minutes: 6\n'
                                    '  max_turns: 12\n',
                'prompts/system.md': 'You are a senior software engineer fixing one issue in '
                                     '`/workspace`. Make a minimal, correct production-code patch '
                                     'and submit it.\n'
                                     '\n'
                                     '## Finish first\n'
                                     '- Call `submit_patch()` exactly once, then give a brief '
                                     'text-only completion. If prompted again after submission, '
                                     'answer in text and make no tool calls.\n'
                                     '- Submit by your 8th tool call at the latest. Submit sooner '
                                     'once the edit and one focused review are complete.\n'
                                     '- Never call `get_status` or call `submit_patch` a second '
                                     'time.\n'
                                     '\n'
                                     '## Fast, focused workflow\n'
                                     '1. Read the issue carefully. Search once for its exact '
                                     'symbol or distinctive phrase using `rg -n -F`. Use the '
                                     'repository layout from `rg --files` only if the named path '
                                     'is absent. If a search fails, adjust the path or query once; '
                                     'do not repeat it unchanged.\n'
                                     '2. Read the smallest useful source range. Do not read the '
                                     'same file/range twice. Use `read_file` directly; never type '
                                     'pseudo tool syntax into a shell command.\n'
                                     '3. Make a minimal production edit by your 4th tool call. If '
                                     'two source reads do not reveal the full design, choose the '
                                     'best evidence-based local fix and submit it instead of '
                                     'broadening the search.\n'
                                     '4. Tests are read-only. Never edit, create, delete, or '
                                     'rename tests or fixtures. Do not create scratch files in the '
                                     'repository. Do not run a test suite; the evaluator verifies '
                                     'the patch. At most, run one short check that does not change '
                                     'files.\n'
                                     '5. Review the changed lines once, then submit. Do not repeat '
                                     'an identical search, read, check, or edit.\n'
                                     '\n'
                                     'Keep the change small and preserve nearby conventions. Do '
                                     'not access the network, install packages, inspect git '
                                     'history, or modify packaging and runner configuration. If a '
                                     'tool call fails, correct it once or move on. Never invent '
                                     'test results.\n'}}
EXPECTED_MODEL = 'gemma-4-31b-it-qat-w4a16-ct'
VARIANTS_DIR = WORK / 'agent_variants'
if VARIANTS_DIR.exists():
    shutil.rmtree(VARIANTS_DIR)
VARIANTS_DIR.mkdir(parents=True, exist_ok=True)
for label, files in VARIANT_FILES.items():
    root = VARIANTS_DIR / label
    root.mkdir(parents=True, exist_ok=True)
    for rel, content in files.items():
        path = Path(rel)
        assert not path.is_absolute() and '..' not in path.parts, rel
        dest = root.joinpath(*path.parts)
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_text(content, encoding='utf-8')
    assert (root / 'agent.yaml').is_file()
    print(label, 'files=', len(files), 'root model=', EXPECTED_MODEL)


## Start the official local Gemma runtime

All candidates use the competition's required model alias; vLLM uses tensor parallelism across all four L4 GPUs.


In [ ]:
import litellm, torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
from swegemma.models.discovery import validate_single_declared_model

litellm.drop_params = True
MODEL_ALIAS = 'gemma-4-31b-it-qat-w4a16-ct'
for label in VARIANT_FILES:
    assert validate_single_declared_model(VARIANTS_DIR / label) == MODEL_ALIAS, label
    discovered = discover_adapters(str(VARIANTS_DIR / label),
                                   adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
    assert not discovered.adapters, (label, sorted(discovered.adapters))

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH), port=8000, host='127.0.0.1',
    tool_call_parser='gemma4', reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'auto',
    gpu_memory_utilization=0.90, enable_auto_tool_choice=True,
    enable_lora=False, tensor_parallel_size=4, startup_timeout=60 * 20,
)
server_instance = VllmServer(vllm_cfg)
server_instance.start()
models = server_instance.create_model_registry(
    aliases=[MODEL_ALIAS], model_prefix='openai/', api_key='EMPTY')
print('Official model ready on four L4s:', server_instance.base_url)


## Paired public-task evaluation

Eight fixed public issues are sampled deterministically across repositories. Each arm gets the same issue IDs and official two-phase patch verification. The result table reports verified resolutions, infrastructure failures, accidental test edits, tool calls, and elapsed time.


In [ ]:
import asyncio, concurrent.futures, random, re, time, yaml
from collections import defaultdict
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator
from swegemma.models import load_tasks

TASKS_PATH = DATA_DIR / 'tasks.jsonl'
tasks = load_tasks(TASKS_PATH)
def choose_public_panel(all_tasks, n=8, seed=4207):
    buckets = defaultdict(list)
    for task in all_tasks:
        buckets[task.repo].append(task)
    repos = sorted(buckets)
    shuffled = {}
    for repo in repos:
        items = sorted(buckets[repo], key=lambda t: t.instance_id)
        rng = random.Random(f'{seed}:{repo}')
        shuffled[repo] = rng.sample(items, len(items))
    selected, offsets = [], {repo: 0 for repo in repos}
    for repo in repos:
        if len(selected) < n and shuffled[repo]:
            selected.append(shuffled[repo][0]); offsets[repo] = 1
    while len(selected) < min(n, len(all_tasks)):
        advanced = False
        for repo in repos:
            if offsets[repo] < len(shuffled[repo]) and len(selected) < n:
                selected.append(shuffled[repo][offsets[repo]])
                offsets[repo] += 1; advanced = True
        if not advanced:
            break
    return selected

PANEL = choose_public_panel(tasks, n=8)
PANEL_IDS = [t.instance_id for t in PANEL]
print('Fixed public panel:', [(t.instance_id, t.repo) for t in PANEL])
assert len(PANEL_IDS) == len(set(PANEL_IDS))
limits, gen_constraints = build_submission_limits()
(WORK / 'agent_eval').mkdir(parents=True, exist_ok=True)
(WORK / 'agent_eval' / 'partial_results.jsonl').write_text('', encoding='utf-8')

def run_sync(coro_or_fn, *args, **kwargs):
    fn = (lambda: coro_or_fn(*args, **kwargs)) if callable(coro_or_fn) else (lambda: coro_or_fn)
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())

def patch_paths(patch):
    return [b for a, b in re.findall(r'^diff --git a/(.*?) b/(.*?)$', patch or '', re.MULTILINE)]

def is_test_path(name):
    p = Path(name)
    return ('tests' in p.parts or p.name.startswith('test_') or
            p.name.endswith('_test.py') or p.name in {'conftest.py', 'pytest.ini'})

def is_infrastructure_error(message):
    return bool(re.search(
        r'SubmissionValidationError|MissingRootConfigError|MultipleRootConfigsError|'
        r'Failed to load skill|ModuleNotFoundError|ConnectionRefusedError|'
        r'CUDA out of memory|Sandbox initialization failed|model (?:was )?not found',
        message or '', re.IGNORECASE))

def evaluate_arm(label):
    root = VARIANTS_DIR / label
    section = yaml.safe_load((root / 'eval_config.yaml').read_text())['evaluation']
    cfg = EvalConfig(
        tasks_path=TASKS_PATH, snapshots_dir=DATA_DIR / 'snapshots',
        results_dir=WORK / 'agent_eval' / label, submission_dir=root,
        models=models, sandbox='subprocess', timeout_seconds=int(section['timeout_seconds']),
        max_time_minutes=float(section['max_time_minutes']),
        max_tool_calls=int(section['max_tool_calls']), max_turns=int(section['max_turns']),
        limits=limits, generation_constraints=gen_constraints,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(compaction_interval=15, overlap_size=2,
            token_threshold=14336, event_retention_size=5),
        graph_dir=str(DATA_DIR / 'graphs'), embeddings_dir=str(DATA_DIR / 'embeddings'),
        wheels_dir=DATA_DIR / 'wheels', verbose=False, display_mode='quiet', concurrency=1,
    )
    evaluator = Evaluator(cfg)
    records = []
    for i, task in enumerate(PANEL, 1):
        start = time.monotonic()
        try:
            result = run_sync(evaluator.evaluate_task, task=task, task_index=i, total_tasks=len(PANEL))
            patch = result.agent_patch or ''
            changed = patch_paths(patch)
            problem = (str(getattr(result, 'agent_error', '') or '') + ' ' +
                       str(getattr(result, 'error', '') or '')).strip()[:600]
            row = {
                'arm': label, 'id': task.instance_id, 'repo': task.repo,
                'resolved': bool(result.resolved), 'patch_chars': len(patch),
                'tool_calls': int(getattr(result, 'tool_calls', 0) or 0),
                'duration_seconds': round(float(getattr(result, 'duration_seconds', time.monotonic()-start) or 0), 1),
                'test_files_touched': int(any(is_test_path(p) for p in changed)),
                'test_exit_code': getattr(result, 'test_exit_code', None),
                'error': problem,
                'infra_error': is_infrastructure_error(problem),
            }
        except Exception as exc:
            problem = f'{type(exc).__name__}: {exc}'[:600]
            row = {'arm': label, 'id': task.instance_id, 'repo': task.repo,
                   'resolved': False, 'patch_chars': 0, 'tool_calls': 0,
                   'duration_seconds': round(time.monotonic()-start, 1),
                   'test_files_touched': 0, 'test_exit_code': None,
                   'error': problem, 'infra_error': True}
        records.append(row)
        print(label, i, '/', len(PANEL), row, flush=True)
        log = WORK / 'agent_eval' / 'partial_results.jsonl'
        log.parent.mkdir(parents=True, exist_ok=True)
        with log.open('a', encoding='utf-8') as stream:
            stream.write(json.dumps(row, ensure_ascii=False) + '\n')
    return records

ARMS = ['lean_v3_8m', 'lean_v4_6m']
RESULTS = {}
for arm in ARMS:
    RESULTS[arm] = evaluate_arm(arm)

def summarize(rows):
    return {
        'resolved': sum(r['resolved'] for r in rows),
        'tasks': len(rows),
        'rate': round(sum(r['resolved'] for r in rows) / max(len(rows), 1), 3),
        'test_files_touched': sum(r['test_files_touched'] for r in rows),
        'infra_errors': sum(r['infra_error'] for r in rows),
        'mean_seconds': round(sum(r['duration_seconds'] for r in rows) / max(len(rows), 1), 1),
        'mean_tool_calls': round(sum(r['tool_calls'] for r in rows) / max(len(rows), 1), 1),
    }

SUMMARY = {arm: summarize(rows) for arm, rows in RESULTS.items()}
print('SUMMARY:', json.dumps(SUMMARY, indent=2))


## Choose and package the measured winner

The package selector uses only local public-task verification outcomes. It excludes genuine infrastructure/configuration errors and test-file edits, but treats model context or turn-budget exhaustion as agent failures. It then chooses the highest resolved count; ties prefer the lean agent, then the shorter runtime. This panel is a development comparison, not Kaggle's hidden public score.


In [ ]:
eligible = [arm for arm in ARMS
            if SUMMARY[arm]['infra_errors'] == 0 and SUMMARY[arm]['test_files_touched'] == 0]
if not eligible:
    raise RuntimeError('All agent arms had infrastructure errors or touched test files; inspect the per-task records.')
tie_order = {'lean_v4_6m': 0, 'lean_v3_8m': 1}
SELECTED = max(eligible, key=lambda arm: (
    SUMMARY[arm]['resolved'], -SUMMARY[arm]['mean_seconds'], -tie_order[arm]))
print('Selected arm:', SELECTED, SUMMARY[SELECTED])

import hashlib, zipfile
selected_files = VARIANT_FILES[SELECTED]
zip_path = WORK / 'submission.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for rel, content in sorted(selected_files.items()):
        archive.writestr(rel, content.encode('utf-8'))
with zipfile.ZipFile(zip_path) as archive:
    assert 'agent.yaml' in archive.namelist() and archive.testzip() is None
    assert len([p for p in archive.namelist() if p in
                {'agent.yaml','agent.yml','root_agent.yaml','root_agent.yml'}]) == 1
experiment = {
    'model': MODEL_ALIAS, 'machine': '4x NVIDIA L4',
    'public_panel_ids': PANEL_IDS, 'arms': SUMMARY, 'selected_arm': SELECTED,
    'submission_zip_sha256': hashlib.sha256(zip_path.read_bytes()).hexdigest(),
    'caveat': 'Eight public development issues are a small local comparison, not Kaggle hidden/public leaderboard score.',
}
(WORK / 'experiment_card.json').write_text(json.dumps(experiment, indent=2) + '\n', encoding='utf-8')
print('Created:', zip_path, 'bytes=', zip_path.stat().st_size)
print('SHA256:', experiment['submission_zip_sha256'])
